In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
import pandas as pd
from omegaconf import OmegaConf
from pathlib import Path
import os
import sys
if ".. " not in sys.path:
    sys.path.append("..")
from src.utils import Metrics

OUTPUT_DIR = Path("..") / "output"
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"

EXPERIMENTS = [
    "compare_methods_linear_deterministic",
    "compare_methods_linear_stochastic",
    "compare_momentum_rank_linear_stochastic",
    "compare_methods_mnist_lenet5",
    # "compare_methods_cifar10_patchmlp",
    "compare_methods_cifar100_patchmlp",
]

In [ ]:
def get_experiment_df(experiment_name):
    dfs_list = []
    for filename in OUTPUT_DIR.glob("*/*"):
        run_dir = Path(filename)
        if not run_dir.is_dir():
            continue  # skip non-directories

        # Load config
        if not os.path.exists(run_dir / "config.yaml"):
            continue
        cfg = OmegaConf.load(run_dir / "config.yaml")

        try:
            cfg.task.name
        except AttributeError:
            continue  # wrong config version, skip run

        if not cfg.experiment.startswith(experiment_name):
            continue  # wrong experiment, skip run

        # Load metrics
        if not os.path.exists(run_dir / "metrics.json"):
            continue
        metrics = Metrics()
        metrics.load(run_dir / "metrics.json")

        # convert metrics to dataframe and add relevant hyperparameters
        df = metrics.to_pandas().astype(float)
        df["Seed"] = cfg.seed
        df["LR"] = cfg.lr
        df["Task"] = cfg.task.name
        df["method_name"] = cfg.method.name
        df["optimizer_name"] = cfg.optimizer.name

        if "oplora" in cfg.optimizer.name:
            df["oplora_iters"] = cfg.oplora_iters
            df["oplora_lmbd"] = cfg.oplora_lmbd
            try:
                df["oplora_rank_mult"] = cfg.oplora_rank_mult
            except:
                df["oplora_rank_mult"] = 1

        # Method column for nice visualization
        if "oplora" in cfg.optimizer.name:
            if "proj" in cfg.optimizer.name:
                df["Method"] = "Proj. PSI-LoRA x " + str(cfg.oplora_iters)
            elif "fista" in cfg.optimizer.name:
                df["Method"] = "FISTA PSI-LoRA x " + str(cfg.oplora_iters)
            elif "scaled" in cfg.optimizer.name:
                df["Method"] = "Scaled PSI-LoRA x " + str(cfg.oplora_iters)
            else:
                df["Method"] = "PSI-LoRA x " + str(cfg.oplora_iters)
        elif "precond_lora" in cfg.optimizer.name:
            df["Method"] = "RPLoRA"
        else:
            if cfg.method.name == "lora":
                df["Method"] = "LoRA"
            elif cfg.method.name == "full":
                df["Method"] = "Full"
            elif cfg.method.name == "svdlora":
                df["Method"] = "SVDLoRA"
            else:
                df["Method"] = cfg.method.name.capitalize()
        
        if "adam" in cfg.optimizer.name:
            df["Optimizer"] = "AdamW"
        else:
            df["Optimizer"] = "SGD"

        dfs_list.append(df)

    assert len(dfs_list) > 0, f"No data found for experiment {experiment_name}"
    return pd.concat(dfs_list).reset_index()

## compare_methods_linear_deterministic

In [ ]:
experiment_name = "compare_methods_linear_deterministic"
experiment_df = get_experiment_df(experiment_name)

In [ ]:
plot_df = experiment_df.copy()

sns_opts = {
    "hue": "Method",
    "hue_order": [
        "SVDLoRA",
        "LoRA",
        "RPLoRA",
        "PSI-LoRA x 1",
        "PSI-LoRA x 2",
        "PSI-LoRA x 8",
        "PSI-LoRA x 32",
        "PSI-LoRA x 128"
    ],
    "style": "Optimizer",
    "style_order": ["SGD ($\\alpha=0$)", "AdamW"],
    # "errorbar": "sd",  # uncomment for faster plots
}
plot_df.loc[plot_df["Optimizer"] == "SGD", "Optimizer"] = "SGD ($\\alpha=0$)"

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.lineplot(ax=ax, data=plot_df, x="epochs", y="test/loss", **sns_opts)

ax.grid()
ax.set_yscale("log")
ax.set_xlabel("Epochs")
ax.set_ylabel("Test Loss")
plt.suptitle("Linear task (Full-batch)")

ax.legend(bbox_to_anchor=(0.5, -0.4), loc="lower center", ncol=4, frameon=True, fontsize=8)
fig.tight_layout()

plt.savefig(PLOT_DIR / (experiment_name + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

## compare_methods_linear_stochastic

In [ ]:
experiment_name = "compare_methods_linear_stochastic"
experiment_df = get_experiment_df(experiment_name)

In [ ]:
plot_df = experiment_df.copy()

sns_opts = {
    "hue": "Method",
    "hue_order": [
        "SVDLoRA",
        "LoRA",
        "RPLoRA",
        "PSI-LoRA x 1",
        "PSI-LoRA x 2",
        "PSI-LoRA x 8",
        "Proj. PSI-LoRA x 1",
        "Proj. PSI-LoRA x 2",
        "Proj. PSI-LoRA x 8"
    ],
    "style": "Optimizer",
    "style_order": ["SGD ($\\alpha=0.75$)", "AdamW"],
    # "errorbar": "sd",
}
plot_df.loc[plot_df["Optimizer"] == "SGD", "Optimizer"] = "SGD ($\\alpha=0.75$)"
# plot_df = plot_df[plot_df["epochs"] >= 10]

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.lineplot(ax=ax, data=plot_df, x="epochs", y="test/loss", **sns_opts)

ax.grid()
ax.set_yscale("log")
ax.set_xlabel("Epochs")
ax.set_ylabel("Test Loss")
plt.suptitle("Linear task (Mini-batch)")

ax.legend(bbox_to_anchor=(0.5, -0.5), loc="lower center", ncol=4, frameon=True, fontsize=8)
fig.tight_layout()

plt.savefig(PLOT_DIR / (experiment_name + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

## compare_momentum_rank_linear_stochastic

In [ ]:
experiment_name = "compare_momentum_rank_linear_stochastic"
experiment_df = get_experiment_df(experiment_name)

In [ ]:
plot_df = experiment_df.copy()

rank_mult_col = r"$\mathcal{M}(\mathbf{G})$ Rank"
sns_opts = {
    "hue": "Method",
    "hue_order": [
        "SVDLoRA",
        "",  # just a hack to use consistent hues
        "",
        "PSI-LoRA x 1",
        "",
        "PSI-LoRA x 8"
    ],
    "style": rank_mult_col,
    "style_order": [
        "$r$",
        "$2r$",
        "$4r$"
    ],
    "palette": "tab10",
    # "errorbar": "sd",
}
plot_df["oplora_rank_mult"] = plot_df["oplora_rank_mult"].fillna("1")  # for svdlora
plot_df[rank_mult_col] = plot_df["oplora_rank_mult"].apply(lambda x: f"${int(x)}r$" if int(x) > 1 else "$r$")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.lineplot(ax=ax, data=plot_df, x="epochs", y="test/loss", **sns_opts)

ax.grid()
ax.set_yscale("log")
ax.set_xlabel("Epochs")
ax.set_ylabel("Test Loss")
plt.suptitle("Linear task (Mini-batch)")
fig.tight_layout()

plt.savefig(PLOT_DIR / (experiment_name + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

## compare_methods_mnist_lenet5

In [ ]:
experiment_name = "compare_methods_mnist_lenet5"
experiment_df = get_experiment_df(experiment_name)

In [ ]:
plot_df = experiment_df.copy()

hue_order = [
    "SVDLoRA",
    "LoRA",
    "RPLoRA",
    "PSI-LoRA x 1",
    "PSI-LoRA x 2",
    "PSI-LoRA x 8",
    "Scaled PSI-LoRA x 1",
    "Scaled PSI-LoRA x 2",
    "Scaled PSI-LoRA x 8",
    # "Proj. OPLoRA x 1", "Proj. OPLoRA x 2", "Proj. OPLoRA x 8",
    # "FISTA OPLoRA x 1", "FISTA OPLoRA x 2", "FISTA OPLoRA x 8",
    "Full",
]

sns_opts = {
    "hue": "Method",
    "hue_order": hue_order,
    "style": "Optimizer",
    "style_order": ["SGD ($\\alpha=0.9$)", "AdamW"],
    # "errorbar": "sd",
}

plot_df.loc[plot_df["Optimizer"] == "SGD", "Optimizer"] = "SGD ($\\alpha=0.9$)"

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
sns.lineplot(ax=ax[0], data=plot_df, x="epochs", y="train/loss", **sns_opts)
sns.lineplot(ax=ax[1], data=plot_df, x="epochs", y="test/accuracy", **sns_opts)

ax[0].grid()
ax[1].grid()
ax[0].set_yscale("log")
# ax[1].set_yscale("log")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Train Loss")
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Test Accuracy")

plt.suptitle("MNIST Task (Batch Size = 64)")
fig.tight_layout()
ax[0].get_legend().remove()
# ax[1].legend(bbox_to_anchor=(1.36, 0.5), loc="center right", frameon=True)

plt.savefig(PLOT_DIR / (experiment_name + PLOT_SUFFIX), bbox_inches="tight")
plt.show()

### Save MNIST logs for table creation

In [ ]:
mnist_df = experiment_df.copy()
mnist_df = mnist_df[mnist_df["Method"].isin(hue_order)].copy()  # Filter out methods not in hue_order

## compare_methods_cifar100_patchmlp

In [ ]:
experiment_name = "compare_methods_cifar100_patchmlp"
experiment_df = get_experiment_df(experiment_name)

In [ ]:
plot_df = experiment_df.copy()

hue_order = [
    "SVDLoRA",
    "LoRA",
    "RPLoRA",
    "PSI-LoRA x 1",
    "PSI-LoRA x 2",
    "PSI-LoRA x 8",
    "Scaled PSI-LoRA x 1",
    "Scaled PSI-LoRA x 2",
    "Scaled PSI-LoRA x 8",
    "Full",
]

sns_opts = {
    "hue": "Method",
    "hue_order": hue_order,
    "style": "Optimizer",
    "style_order": ["SGD ($\\alpha=0.9$)", "AdamW"],
    # "errorbar": "sd",
}

plot_df.loc[plot_df["Optimizer"] == "SGD", "Optimizer"] = "SGD ($\\alpha=0.9$)"

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
sns.lineplot(ax=ax[0], data=plot_df, x="epochs", y="train/loss", **sns_opts)
sns.lineplot(ax=ax[1], data=plot_df, x="epochs", y="test/accuracy", **sns_opts)
ax[0].grid()
ax[1].grid()
ax[0].set_yscale("log")
# ax[1].set_yscale("log")
ax[0].set_xlabel("Epochs")
ax[0].set_ylabel("Train Loss")
ax[1].set_xlabel("Epochs")
ax[1].set_ylabel("Test Accuracy")

plt.suptitle("CIFAR-100 Task (Batch Size = 64)")
fig.tight_layout()
ax[0].get_legend().remove()
# ax[1].legend(bbox_to_anchor=(1.36, 0.5), loc="center right", frameon=True)

plt.savefig(PLOT_DIR / (experiment_name + PLOT_SUFFIX), bbox_inches="tight")
plt.show()

In [ ]:
cifar100_df = experiment_df.copy()
cifar100_df = cifar100_df[cifar100_df["Method"].isin(hue_order)].copy()

# Create Tables for MNIST and CIFAR-100

### Table creation method (auto bold for first, underline for second)

In [ ]:
def _emphasize_best_second_mean_std(mean_series, std_series=None, precision=2):
    """Format mean values with best bolded and 2nd-best underlined.

    If std_series is provided, append std dev in scriptsize:
      mean{\scriptsize $\pm$std}

    Emphasis is applied to the mean only.
    """
    s_mean = mean_series.astype(float)
    best = s_mean.max()

    # second best (strictly less than best); if all equal, skip underline
    s_less = s_mean[s_mean < best]
    second = s_less.max() if len(s_less) else None

    if std_series is None:
        s_std = None
    else:
        s_std = std_series.astype(float)

    def fmt(idx, v_mean):
        if pd.isna(v_mean):
            return ""

        mean_txt = f"{v_mean:.{precision}f}"
        if v_mean == best:
            mean_txt = f"\\textbf{{{mean_txt}}}"
        elif second is not None and v_mean == second:
            mean_txt = f"\\underline{{{mean_txt}}}"

        if s_std is None:
            return mean_txt

        v_std = s_std.loc[idx]
        if pd.isna(v_std) or v_std == 0:
            return mean_txt

        std_txt = f"{v_std:.{precision}f}"
        return mean_txt + r"{\scriptsize $\pm$" + std_txt + "}"

    return pd.Series({idx: fmt(idx, v) for idx, v in s_mean.items()})


def create_metric_tables(data_df, metric_col="test/accuracy", as_percent=True, caption=None, label=None):
    metric_per_baseline_seed = data_df.groupby(["Method", "Optimizer", "Seed"])[metric_col]

    # Get per-seed metrics
    top_metric_per_baseline_seed = metric_per_baseline_seed.max()
    mean_metric_per_baseline_seed = metric_per_baseline_seed.mean()
    last_metric_per_baseline_seed = metric_per_baseline_seed.last()

    # Average + std over seeds
    group_keys = ["Method", "Optimizer"]

    top_mean = top_metric_per_baseline_seed.groupby(group_keys).mean().rename("Top")
    top_std = top_metric_per_baseline_seed.groupby(group_keys).std().rename("Top_std")

    mean_mean = mean_metric_per_baseline_seed.groupby(group_keys).mean().rename("Mean")
    mean_std = mean_metric_per_baseline_seed.groupby(group_keys).std().rename("Mean_std")

    last_mean = last_metric_per_baseline_seed.groupby(group_keys).mean().rename("Last")
    last_std = last_metric_per_baseline_seed.groupby(group_keys).std().rename("Last_std")

    # For display in notebook
    top_metric_per_baseline = top_mean.reset_index().rename(columns={"Top": metric_col})
    mean_metric_per_baseline = mean_mean.reset_index().rename(columns={"Mean": metric_col})
    last_metric_per_baseline = last_mean.reset_index().rename(columns={"Last": metric_col})

    print(f"Top {metric_col} per method:")
    display(top_metric_per_baseline.sort_values(metric_col, ascending=False))
    print(f"Mean {metric_col} per method:")
    display(mean_metric_per_baseline.sort_values(metric_col, ascending=False))
    print(f"Last {metric_col} per method:")
    display(last_metric_per_baseline.sort_values(metric_col, ascending=False))

    latex_df = pd.concat([top_mean, top_std, mean_mean, mean_std, last_mean, last_std], axis=1).reset_index()

    if as_percent:
        for c in ["Top", "Top_std", "Mean", "Mean_std", "Last", "Last_std"]:
            latex_df[c] = 100.0 * latex_df[c]
        latex_df = latex_df.rename(columns={"Top": "Top (\\%)", "Mean": "Mean (\\%)", "Last": "Last (\\%)"})
    else:
        latex_df = latex_df.rename(columns={"Top": "Top", "Mean": "Mean", "Last": "Last"})

    # Order rows by Top (or first metric column)
    metric_cols = [c for c in latex_df.columns if c not in ["Method", "Optimizer", "Top_std", "Mean_std", "Last_std"]]
    latex_df = latex_df.sort_values(metric_cols[0], ascending=False).reset_index(drop=True)

    # Create LaTeX-formatted copy with best/2nd-best emphasized per column,
    # and append std dev in scriptsize.
    latex_df_latex = latex_df.copy()

    std_map = {
        metric_cols[0]: "Top_std" if metric_cols[0].startswith("Top") else None,
        metric_cols[1]: "Mean_std" if metric_cols[1].startswith("Mean") else None,
        metric_cols[2]: "Last_std" if metric_cols[2].startswith("Last") else None,
    }

    for c in metric_cols:
        std_c = std_map.get(c)
        latex_df_latex[c] = _emphasize_best_second_mean_std(latex_df[c], latex_df[std_c] if std_c else None, precision=2)

    latex_df_latex = latex_df_latex[["Method", "Optimizer"] + metric_cols]

    latex_str = latex_df_latex.to_latex(
        index=False,
        escape=False,
        caption=caption,
        label=label,
    )

    print("\nLaTeX table (best=bold, 2nd=underline; std in scriptsize):\n")
    print(latex_str)

    return latex_str

In [ ]:
mnist_latex_table = create_metric_tables(
    mnist_df,
    metric_col="test/accuracy",
    as_percent=True,
    caption="MNIST (LeNet5) test accuracy summary.",
    label="tab:mnist_lenet5_results"
)

In [ ]:
cifar100_latex_table = create_metric_tables(
    cifar100_df,
    metric_col="test/accuracy",
    as_percent=True,
    caption="CIFAR-100 (PatchMLP) test accuracy summary.",
    label="tab:cifar100_results"
)